# Exploring Geodatabase Files

This notebook demonstrates how to explore and inventory ESRI File Geodatabase (.gdb) files using Python GIS libraries.

**What you'll learn:**
- List all layers in a geodatabase
- Inspect layer schemas and geometry types
- Examine coordinate reference systems (CRS)
- Preview feature data

In [1]:
# Import required libraries
import fiona
import geopandas as gpd
import pandas as pd
from pathlib import Path

## 1. Configure Geodatabase Path

Set the path to your geodatabase file. The default path points to the example data.

In [3]:
# Configure the path to your geodatabase
GDB_PATH = "../data-extracted/RVO_NW_I_GGM.gdb"

# Verify the path exists
gdb_path = Path(GDB_PATH)
if gdb_path.exists():
    print(f"Geodatabase found: {gdb_path.absolute()}")
else:
    print(f"WARNING: Geodatabase not found at {GDB_PATH}")
    print("Please update GDB_PATH to point to your geodatabase file.")

Geodatabase found: /app/notebooks/../data-extracted/RVO_NW_I_GGM.gdb


## 2. List All Layers

Use Fiona to list all layers available in the geodatabase.

In [4]:
# List all layers in the geodatabase
layers = fiona.listlayers(GDB_PATH)
print(f"Total layers found: {len(layers)}\n")

# Display first 20 layers
for i, layer in enumerate(sorted(layers)[:20]):
    print(f"{i+1:3}. {layer}")

if len(layers) > 20:
    print(f"\n... and {len(layers) - 20} more layers")

Total layers found: 79

  1. Borehole_locations
  2. Buried_channels
  3. Cables
  4. Downhole_Geophysical_Measurements__BGL_
  5. E3_MTDs_Above_60mBSF
  6. E3_MTDs_Below_60mBSF
  7. E_1_Unit_Channels
  8. E_4_Potential_Glaciotectonised_Sediments
  9. E_5_Potential_Glaciotectonised_Sediments
 10. Faults
 11. Geophysical_Survey_Magnetometer_Contacts
 12. Geophysical_Survey_Priority_Seismic_Lines
 13. Geophysical_Survey_SSS_Contacts
 14. Geophysical_Survey_Sediment_Primary
 15. IJV_tielines
 16. Infrastructure_As_found_Infrastructure_Arc
 17. Mainland
 18. MobileSediments_SubcropFeatures
 19. MobileSediments_Thickness_Contours_1m
 20. NW_Site_I_Investigation_Area

... and 59 more layers


## 3. Inspect Layer Details

Examine the schema, geometry type, and CRS for each layer.

In [5]:
def get_layer_info(gdb_path, layer_name):
    """Get detailed information about a single layer."""
    try:
        with fiona.open(gdb_path, layer=layer_name) as src:
            return {
                'name': layer_name,
                'geometry_type': src.schema.get('geometry', 'None'),
                'crs': str(src.crs) if src.crs else 'Unknown',
                'feature_count': len(src),
                'fields': list(src.schema['properties'].keys()),
                'bounds': src.bounds if len(src) > 0 else None,
                'status': 'OK'
            }
    except Exception as e:
        return {
            'name': layer_name,
            'status': 'Error',
            'error': str(e)
        }

In [6]:
# Collect information for all layers
layer_info_list = []

for layer_name in sorted(layers):
    info = get_layer_info(GDB_PATH, layer_name)
    layer_info_list.append(info)

# Create a summary DataFrame
successful_layers = [l for l in layer_info_list if l['status'] == 'OK']
failed_layers = [l for l in layer_info_list if l['status'] == 'Error']

print(f"Successfully loaded: {len(successful_layers)} layers")
print(f"Failed to load: {len(failed_layers)} layers (likely raster or system tables)")

Successfully loaded: 79 layers
Failed to load: 0 layers (likely raster or system tables)


In [ ]:
# Create a summary table
summary_data = []
for info in successful_layers:
    summary_data.append({
        'Layer': info['name'],
        'Geometry': info['geometry_type'] or 'Table',
        'Features': info['feature_count'],
        'Fields': len(info['fields']),
        'CRS': info['crs'][:30] + '...' if len(info['crs']) > 30 else info['crs']
    })

summary_df = pd.DataFrame(summary_data)
summary_df

## 4. Group Layers by Geometry Type

Organize layers by their geometry type to understand the data structure.

In [ ]:
# Group by geometry type
by_geometry = {}
for info in successful_layers:
    geom_type = info['geometry_type'] or 'Table'
    if geom_type not in by_geometry:
        by_geometry[geom_type] = []
    by_geometry[geom_type].append(info)

# Display summary
for geom_type, layers_list in sorted(by_geometry.items()):
    total_features = sum(l['feature_count'] for l in layers_list)
    print(f"\n{geom_type}: {len(layers_list)} layers, {total_features:,} total features")
    print("-" * 50)
    for info in layers_list[:5]:
        print(f"  - {info['name']} ({info['feature_count']:,} features)")
    if len(layers_list) > 5:
        print(f"  ... and {len(layers_list) - 5} more")

## 5. Examine a Specific Layer

Load and inspect a single layer in detail.

In [ ]:
# Choose a layer to examine (change this to explore different layers)
LAYER_TO_EXAMINE = successful_layers[0]['name'] if successful_layers else None

if LAYER_TO_EXAMINE:
    print(f"Examining layer: {LAYER_TO_EXAMINE}")

In [ ]:
# Load the layer using GeoPandas
if LAYER_TO_EXAMINE:
    gdf = gpd.read_file(GDB_PATH, layer=LAYER_TO_EXAMINE)
    
    print(f"Shape: {gdf.shape[0]} rows x {gdf.shape[1]} columns")
    print(f"Geometry type: {gdf.geometry.geom_type.unique()}")
    print(f"CRS: {gdf.crs}")
    print(f"\nBounds:")
    print(f"  minx: {gdf.total_bounds[0]:.6f}")
    print(f"  miny: {gdf.total_bounds[1]:.6f}")
    print(f"  maxx: {gdf.total_bounds[2]:.6f}")
    print(f"  maxy: {gdf.total_bounds[3]:.6f}")

In [ ]:
# Display column information
if LAYER_TO_EXAMINE:
    print("Columns and data types:\n")
    for col in gdf.columns:
        dtype = gdf[col].dtype
        non_null = gdf[col].notna().sum()
        print(f"  {col}: {dtype} ({non_null} non-null values)")

In [ ]:
# Preview the data
if LAYER_TO_EXAMINE:
    # Show first few rows (excluding geometry for readability)
    display_cols = [c for c in gdf.columns if c != 'geometry'][:10]
    gdf[display_cols].head()

## 6. Export Layer Inventory

Save the layer inventory to JSON for later use.

In [ ]:
import json

# Prepare inventory data
inventory = {
    'geodatabase': GDB_PATH,
    'total_layers': len(layers),
    'vector_layers_count': len(successful_layers),
    'raster_layers_count': len(failed_layers),
    'total_features': sum(l['feature_count'] for l in successful_layers),
    'by_geometry': {
        geom: [l['name'] for l in layers_list]
        for geom, layers_list in by_geometry.items()
    },
    'layers': successful_layers
}

# Save to file
output_file = 'layer_inventory.json'
with open(output_file, 'w') as f:
    json.dump(inventory, f, indent=2, default=str)

print(f"Inventory saved to: {output_file}")

## Next Steps

Now that you've explored the geodatabase structure, you can:

1. **Visualize layers** - See `02_visualize_layers.ipynb`
2. **Perform spatial analysis** - See `03_spatial_analysis.ipynb`
3. **Create a dashboard** - Follow the instructions in `CLAUDE.md`